# HB-Adaptive Prior-Learning Diagnostics

This notebook recreates the PDF summary for the current `model-verification` branch model. It uses the shared package model `observers.models.hb_adaptive_confidence.HBAdaptiveConfidenceObserver`, the shared data CSV, and the committed fitted parameters in `hierarchical/results/fits/comparison/hb_adaptive`.

Main question: prior learning at each coherence level (`0.06`, `0.12`, `0.24`) across prior standard deviations (`10`, `20`, `40`, `80`).


## Inputs and Outputs

Inputs used by the script:

- `../../.. /data/data01_direction4priors.csv` from the shared hierarchical folder.
- `../../.. /results/fits/comparison/hb_adaptive/subject<N>.json` for the fitted HB-Adaptive parameters.
- `../../.. /observers/` for the model implementation.

Outputs written locally in this folder:

- `hb_adaptive_all_participants_learning_diagnostics.pdf`
- `results/hb_adaptive_learning_anova_fixed_subject.csv`
- `results/hb_adaptive_learning_by_subject_condition_phase.csv`
- `results/hb_adaptive_block_alignment_summary.csv`
- `results/hb_adaptive_fit_summary.csv`

Plot scale convention: learned prior-width plots use the same visible prior-SD scale, `0-86` degrees, with ticks at the experimental prior widths `10`, `20`, `40`, and `80` degrees. Participant trajectory pages also show the true prior schedule as a bold black step line over colored prior-context bands.


In [1]:
from pathlib import Path
import subprocess
import sys

HERE = Path.cwd()
print('working folder:', HERE)
assert (HERE / 'build_hb_adaptive_learning_pdf.py').exists(), 'Run this notebook from hierarchical/experiments/salma'


working folder: C:\Users\HP\Desktop\baysiean hyperprior\hierarchical\experiments\salma


## Rebuild the PDF

This cell replays the fitted HB-Adaptive model for all 12 participants, computes trial-level learned confidence and learned prior width, summarizes them by coherence, prior width, and block phase, and writes the PDF.


In [2]:
subprocess.run([sys.executable, 'build_hb_adaptive_learning_pdf.py'], check=True)


CompletedProcess(args=['C:\\Python314\\python.exe', 'build_hb_adaptive_learning_pdf.py'], returncode=0)

## Inspect the Quantification

The ANOVA-style table uses subject-fixed effects and tests `coherence`, `prior_std`, and their interaction on the learned hidden variables. This is a post-fit diagnostic, not a separate new model fit.


In [3]:
import pandas as pd

anova = pd.read_csv('results/hb_adaptive_learning_anova_fixed_subject.csv')
display(anova)

summary = pd.read_csv('results/hb_adaptive_learning_by_subject_condition_phase.csv')
display(summary.head())


,measure,model,effect,df_effect,df_error,F,p
0,mean_believed_alpha,subject_fixed + coherence + prior_std,coherence,2,127,0.002370,0.997633
1,mean_believed_alpha,subject_fixed + coherence + prior_std,prior_std,3,127,489.966310,0.000000
2,mean_believed_alpha,subject_fixed + coherence * prior_std,coherence:prior_std,6,121,0.004625,1.000000
3,mean_believed_sd,subject_fixed + coherence + prior_std,coherence,2,127,0.008730,0.991309
4,mean_believed_sd,subject_fixed + coherence + prior_std,prior_std,3,127,1413.724166,0.000000
5,mean_believed_sd,subject_fixed + coherence * prior_std,coherence:prior_std,6,121,0.031908,0.999858


,subject_id,motion_coherence,prior_std,block_phase,n_trials,mean_believed_sd,mean_believed_alpha,median_believed_sd,median_believed_alpha
0,1,0.06,10,early,290,10.725586,0.931591,10.085084,0.955689
1,1,0.06,10,late,288,10.137773,0.954521,10.166392,0.962390
2,1,0.06,10,middle,278,10.177276,0.954395,10.205367,0.962174
3,1,0.06,20,early,297,17.579943,0.874197,17.907357,0.904725
4,1,0.06,20,late,262,17.944296,0.911591,18.046060,0.928849


## How to Present Salma's Result

Start with the model question: does the HB-Adaptive observer learn the task's prior context across blocks, and is that learning mainly explained by prior width or by sensory coherence?

Use Page 2 and Page 3 as the main evidence. Page 2 shows learned prior confidence `E[alpha]`; Page 3 shows learned prior width. The clean result is that hidden prior learning changes strongly across true prior SDs `10`, `20`, `40`, and `80` degrees, while the three coherence panels are nearly overlapping. That is the expected signature if feedback-driven prior learning captures the block prior and coherence mostly acts through sensory uncertainty/prediction rather than through the hidden prior update.

Then use Page 4 as the validity check: each session/run block contains one prior width, so the block labels are aligned. Use participant pages only as examples showing the same mechanism over chronological trials: the black step and colored bands are the true prior schedule, and the learned SD should follow it with a lag because belief is updated after feedback.


## Page Guide

- Page 1: ANOVA-style quantification and model summary.
- Page 2: learned prior confidence `E[alpha]` across prior widths, split by coherence.
- Page 3: learned prior SD across prior widths, split by coherence, fixed to the `0-86` degree prior-width scale.
- Page 4: block alignment diagnostic.
- Pages 5-16: one participant per page, with trajectories and observed-vs-predicted response distributions. The trajectory panel uses the same prior-width scale and highlights true prior blocks with colored bands plus a bold black step line.
